# Multivariate HAL Grid Sensitivity Study

This notebook studies whether changing the multivariate quadrature grid hurts estimator performance. The training data and HAL basis knots stay fixed; only the midpoint quadrature grid used for normalization changes.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
import sys

from IPython.display import display

root = pathlib.Path('.').resolve().parents[1]
sys.path.append(str(root))

np.random.seed(123)

from src.haldensity.multivariate import (
    BivariateTruncatedNormal,
    MultvarHAL,
    extract_term_type_metrics,
)

/Users/houyilong/GitHub/HALDensity/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Study Setup

We generate one bivariate truncated-normal training sample and one independent validation sample. The basis knots come from the training sample and remain unchanged across fits. Only `quadrature_points_per_dim` changes.

In [2]:
dgp = BivariateTruncatedNormal(
    mean=(0.42, 0.58),
    covariance=[[0.018, 0.008], [0.008, 0.022]],
    lower=(0.0, 0.0),
    upper=(1.0, 1.0),
)
support = dgp.bounds

n_train = 80
n_validation = 1500
train_data = dgp.generate_samples(n_train, seed=123)
validation_data = dgp.generate_samples(n_validation, seed=456)

eval_points_per_dim = 120
eval_axis_x1 = np.linspace(support[0][0], support[0][1], eval_points_per_dim)
eval_axis_x2 = np.linspace(support[1][0], support[1][1], eval_points_per_dim)
eval_mesh_x1, eval_mesh_x2 = np.meshgrid(eval_axis_x1, eval_axis_x2, indexing='xy')
eval_grid = np.column_stack([eval_mesh_x1.ravel(), eval_mesh_x2.ravel()])
true_density_eval = dgp.compute_density(eval_grid).reshape(eval_mesh_x1.shape)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(train_data['x1'], train_data['x2'], s=16, alpha=0.55)
ax.set_title('Training sample used for all fits')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
plt.show()

display(train_data.head())
pd.Series(
    {
        'n_train': n_train,
        'n_validation': n_validation,
        'support': support,
        'evaluation_grid_points': eval_grid.shape[0],
    },
    name='value',
)

,x1,x2
0,0.553733,0.686491
1,0.270214,0.422330
2,0.275505,0.496601
3,0.439518,0.700472
4,0.480295,0.600430


n_train                                         80
n_validation                                  1500
support                   ((0.0, 1.0), (0.0, 1.0))
evaluation_grid_points                       14400
Name: value, dtype: object

## Helper Functions

The metrics below measure both numerical behavior and statistical fit quality:

- `validation_avg_loglik`: average holdout log-likelihood on fresh data.
- `fine_midpoint_integral`: a finer-grid normalization check.
- `pointwise_mae`, `pointwise_rmse`, `pointwise_max_abs_error`: dense-grid errors against the known true density.

This notebook now sets `normalizer='midpoint'` explicitly so it continues to isolate tensor-product midpoint-grid effects even though the package default may choose another normalizer automatically.

Because the support is `[0, 1]^2`, the pointwise errors are easy to compare across grid choices.

In [3]:
def average_log_likelihood(model: MultvarHAL, data: pd.DataFrame) -> float:
    density = np.clip(model.get_density_at_points(data), 1e-12, None)
    return float(np.mean(np.log(density)))


def selected_basis_index_set(model: MultvarHAL) -> set[int]:
    return {int(index) for index in model.selected_basis_['basis_index'].tolist()}


def fit_grid_configuration(
    order: int,
    quadrature_points_per_dim: int,
    norm_constraint: float,
) -> tuple[MultvarHAL, np.ndarray, dict[str, float | int | str]]:
    model = MultvarHAL(
        k=order,
        norm_constraint=norm_constraint,
        support=support,
        quadrature_points_per_dim=quadrature_points_per_dim,
        solver='MOSEK',
        use_secondary_solver=True,
        selection_tol=1e-4,
        normalizer='midpoint',
    ).fit(train_data)

    fitted_density_eval = model.get_density_at_points(eval_grid).reshape(eval_mesh_x1.shape)
    pointwise_abs_error = np.abs(fitted_density_eval - true_density_eval)
    pointwise_sq_error = (fitted_density_eval - true_density_eval) ** 2

    row = model.summary()
    row['quadrature_points_per_dim'] = quadrature_points_per_dim
    row['validation_avg_loglik'] = average_log_likelihood(model, validation_data)
    row['fine_midpoint_integral'] = model.normalization_integral(points_per_dim=90)
    row['pointwise_mae'] = float(np.mean(pointwise_abs_error))
    row['pointwise_rmse'] = float(np.sqrt(np.mean(pointwise_sq_error)))
    row['pointwise_max_abs_error'] = float(np.max(pointwise_abs_error))
    row.update(extract_term_type_metrics(model.selected_basis_summary(), prefix='selected'))
    return model, fitted_density_eval, row

## Run The Grid Study

We compare a coarse-to-fine sequence of quadrature grids for `k = 0`, `k = 1`, and `k = 2`. The basis width stays fixed within each order because the training data do not change.

In [4]:
MODEL_ORDERS = [0, 1, 2]
NORM_CONSTRAINTS = {0: 8.0, 1: 50.0, 2: 200.0}
QUADRATURE_POINTS_PER_DIM = [6, 10, 14, 20, 28, 40]

models_by_order: dict[int, dict[int, MultvarHAL]] = {}
surfaces_by_order: dict[int, dict[int, np.ndarray]] = {}
summary_rows: list[dict[str, float | int | str]] = []

for order in MODEL_ORDERS:
    models_by_order[order] = {}
    surfaces_by_order[order] = {}

    for quadrature_points_per_dim in QUADRATURE_POINTS_PER_DIM:
        model, fitted_density_eval, row = fit_grid_configuration(
            order=order,
            quadrature_points_per_dim=quadrature_points_per_dim,
            norm_constraint=NORM_CONSTRAINTS[order],
        )
        models_by_order[order][quadrature_points_per_dim] = model
        surfaces_by_order[order][quadrature_points_per_dim] = fitted_density_eval
        summary_rows.append(row)

grid_study_df = (
    pd.DataFrame(summary_rows)
    .sort_values(['k', 'quadrature_points_per_dim'])
    .reset_index(drop=True)
)

display(
    grid_study_df[
        [
            'k',
            'quadrature_points_per_dim',
            'quadrature_grid_size',
            'solver_used',
            'fit_time_seconds',
            'train_loglik',
            'validation_avg_loglik',
            'fine_midpoint_integral',
            'pointwise_mae',
            'pointwise_rmse',
            'pointwise_max_abs_error',
            'selected_basis_count',
            'selected_nonintercept',
            'selected_poly_count',
            'selected_section_1_count',
            'selected_full_count',
        ]
    ]
)

,k,quadrature_points_per_dim,quadrature_grid_size,solver_used,fit_time_seconds,train_loglik,validation_avg_loglik,fine_midpoint_integral,pointwise_mae,pointwise_rmse,pointwise_max_abs_error,selected_basis_count,selected_nonintercept,selected_poly_count,selected_section_1_count,selected_full_count
0,0,6,36,ECOS,0.023782,172.368753,2.004485,4.982550,4.123362,10.087496,41.464160,8,7,1,3,4
1,0,10,100,ECOS,0.040151,142.467409,1.515721,2.053233,1.344126,3.181807,29.025111,12,11,1,7,4
2,0,14,196,ECOS,0.091983,123.208599,1.221270,1.628224,1.022821,2.375092,19.143802,12,11,1,8,3
3,0,20,400,ECOS,0.245160,119.786940,1.076607,1.258565,0.647661,1.241918,12.633366,18,17,1,11,6
4,0,28,784,ECOS,0.591169,115.622501,1.026879,1.159926,0.581936,1.028975,9.615243,18,17,1,11,6
5,0,40,1600,ECOS,0.741763,112.389487,0.996169,1.123087,0.514245,0.846850,6.023984,19,18,1,11,7
6,1,6,36,ECOS,0.034529,108.872421,1.155533,1.102003,0.354774,0.605208,4.609023,10,9,4,6,0
7,1,10,100,ECOS,0.066912,108.244519,1.088114,1.055957,0.391322,0.806962,10.317717,9,8,4,5,0
8,1,14,196,ECOS,0.132505,106.425280,1.077655,1.028974,0.367498,0.745758,9.593302,12,11,4,8,0
9,1,20,400,ECOS,0.399598,105.244169,1.064699,1.012797,0.347933,0.658324,6.038223,10,9,4,6,0


## Metric Curves

These plots show where the fit appears to stabilize as the quadrature grid becomes finer.

In [5]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5), constrained_layout=True)
metric_specs = [
    ('pointwise_rmse', 'Pointwise RMSE'),
    ('validation_avg_loglik', 'Validation Avg Log-Likelihood'),
    ('fine_midpoint_integral', 'Fine-Grid Integral Check'),
    ('fit_time_seconds', 'Fit Time (seconds)'),
]

for order in MODEL_ORDERS:
    subset = grid_study_df.loc[grid_study_df['k'].eq(order)].sort_values('quadrature_points_per_dim')
    label = f'k = {order}'

    for ax, (metric, title) in zip(axes, metric_specs):
        ax.plot(
            subset['quadrature_points_per_dim'],
            subset[metric],
            marker='o',
            linewidth=2,
            label=label,
        )
        ax.set_title(title)
        ax.set_xlabel('quadrature_points_per_dim')
        ax.grid(alpha=0.3)

axes[0].set_ylabel('error')
axes[1].set_ylabel('avg log-likelihood')
axes[2].set_ylabel('integral')
axes[2].axhline(1.0, color='black', linestyle='--', linewidth=1)
axes[3].set_ylabel('seconds')
axes[0].legend(loc='best')

plt.show()

## Selection Stability Versus The Finest Grid

To see whether a coarse quadrature grid changes the fitted structure, we compare the selected basis set at each grid against the finest grid within the same order.

In [6]:
stability_rows = []
reference_grid = max(QUADRATURE_POINTS_PER_DIM)

for order in MODEL_ORDERS:
    reference_basis = selected_basis_index_set(models_by_order[order][reference_grid])

    for quadrature_points_per_dim in QUADRATURE_POINTS_PER_DIM:
        current_basis = selected_basis_index_set(models_by_order[order][quadrature_points_per_dim])
        union = reference_basis | current_basis
        intersection = reference_basis & current_basis
        jaccard = 1.0 if not union else len(intersection) / len(union)

        stability_rows.append(
            {
                'k': order,
                'quadrature_points_per_dim': quadrature_points_per_dim,
                'reference_grid': reference_grid,
                'selected_basis_jaccard_vs_finest': jaccard,
                'selected_basis_overlap_count': len(intersection),
                'reference_selected_count': len(reference_basis),
                'current_selected_count': len(current_basis),
            }
        )

stability_df = pd.DataFrame(stability_rows)
display(stability_df)

fig, ax = plt.subplots(figsize=(7, 4))
for order in MODEL_ORDERS:
    subset = stability_df.loc[stability_df['k'].eq(order)].sort_values('quadrature_points_per_dim')
    ax.plot(
        subset['quadrature_points_per_dim'],
        subset['selected_basis_jaccard_vs_finest'],
        marker='o',
        linewidth=2,
        label=f'k = {order}',
    )

ax.set_title('Selected-basis stability against the finest grid')
ax.set_xlabel('quadrature_points_per_dim')
ax.set_ylabel('Jaccard overlap')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend(loc='best')
plt.show()

,k,quadrature_points_per_dim,reference_grid,selected_basis_jaccard_vs_finest,selected_basis_overlap_count,reference_selected_count,current_selected_count
0,0,6,40,0.173913,4,19,8
1,0,10,40,0.148148,4,19,12
2,0,14,40,0.148148,4,19,12
3,0,20,40,0.088235,3,19,18
4,0,28,40,0.275862,8,19,18
5,0,40,40,1.000000,19,19,19
6,1,6,40,0.285714,4,8,10
7,1,10,40,0.416667,5,8,9
8,1,14,40,0.250000,4,8,12
9,1,20,40,0.384615,5,8,10


## Surface Snapshots

For a visual check, we compare the true surface with coarse, medium, and fine quadrature grids for each order `k = 0`, `k = 1`, and `k = 2`. The overlaid white contours are always the true density.

In [7]:
visual_grids = [min(QUADRATURE_POINTS_PER_DIM), 20, max(QUADRATURE_POINTS_PER_DIM)]

fig, axes = plt.subplots(len(MODEL_ORDERS), 4, figsize=(24, 5 * len(MODEL_ORDERS)), constrained_layout=True)
if len(MODEL_ORDERS) == 1:
    axes = np.array([axes])

for row_idx, visual_order in enumerate(MODEL_ORDERS):
    truth_contour = axes[row_idx, 0].contourf(
        eval_mesh_x1,
        eval_mesh_x2,
        true_density_eval,
        levels=18,
        cmap='viridis',
    )
    axes[row_idx, 0].scatter(train_data['x1'], train_data['x2'], s=6, color='black', alpha=0.20)
    axes[row_idx, 0].set_title(f'Truth (reference for k = {visual_order})')
    axes[row_idx, 0].set_xlabel('x1')
    axes[row_idx, 0].set_ylabel('x2')

    for ax, quadrature_points_per_dim in zip(axes[row_idx, 1:], visual_grids):
        ax.contourf(
            eval_mesh_x1,
            eval_mesh_x2,
            surfaces_by_order[visual_order][quadrature_points_per_dim],
            levels=18,
            cmap='viridis',
        )
        ax.contour(
            eval_mesh_x1,
            eval_mesh_x2,
            true_density_eval,
            levels=8,
            colors='white',
            linewidths=0.7,
        )
        ax.scatter(train_data['x1'], train_data['x2'], s=6, color='black', alpha=0.20)
        ax.set_title(
            f'k = {visual_order}, grid = {quadrature_points_per_dim}\\n'
            f'({quadrature_points_per_dim ** 2} quadrature points)'
        )
        ax.set_xlabel('x1')
        ax.set_ylabel('x2')

fig.colorbar(truth_contour, ax=axes, shrink=0.85)
plt.show()

## How To Read This Notebook

If coarser grids materially hurt performance, you should usually see at least one of the following:

- worse holdout log-likelihood,
- larger pointwise RMSE or max error,
- a normalization check farther from `1`,
- unstable selected-basis overlap relative to the finest grid.

If those curves flatten out as the grid gets finer, then beyond that point the extra quadrature resolution is mostly a computational cost rather than a statistical gain.